# 04 · Descubrimiento y verificación de la web oficial

La SCVS no publica la web de las empresas y los buscadores prohíben la
consulta automatizada. Las candidatas se derivan del nombre, se descartan por
DNS y solo entonces se visita la portada.

> **Una sola fuente de verdad.** Las funciones de este notebook viven en
> `src/etl/`, se prueban con pytest y son las mismas que corre `etl` en la
> terminal y en CI. Aquí se las llama y se mira el resultado: así el notebook
> no se desactualiza cuando se corrige el código.

In [1]:
# El paquete `etl` se instala con `uv sync` en etl/.venv. Si el kernel del
# notebook no es ese entorno, se agrega src/ a la ruta como respaldo.
import sys
from pathlib import Path

ETL = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
try:
    import etl  # noqa: F401
except ModuleNotFoundError:
    sys.path.insert(0, str(ETL / "src"))

import json

import pandas as pd

from etl import config
from etl.comun import configurar_registro

configurar_registro()
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 180)

ANIO = config.ANIO_POR_DEFECTO
ANIO

2025

## 1. Del nombre al dominio candidato

La razón social suele traer el nombre comercial después de la forma jurídica:
«CONSORCIO ECUATORIANO DE TELECOMUNICACIONES S.A. CONECEL». Ese nombre
comercial es el mejor candidato.

In [2]:
from etl.comun import partes_nombre, tokens_distintivos
from etl.pasos import urls

for nombre in ["PROCESADORA NACIONAL DE ALIMENTOS C.A. PRONACA",
               "CORPORACION FAVORITA C.A.",
               "SOCIEDAD NACIONAL DE GALAPAGOS CA"]:
    base, comercial = partes_nombre(nombre)
    print(f"{nombre}\n  razón social: {base}\n  comercial: {comercial or '—'}"
          f"\n  tokens: {tokens_distintivos(nombre)}\n  candidatos: {urls.candidatos_de(nombre)[:6]}\n")

PROCESADORA NACIONAL DE ALIMENTOS C.A. PRONACA
  razón social: PROCESADORA NACIONAL DE ALIMENTOS
  comercial: PRONACA
  tokens: ['PRONACA', 'ALIMENTOS']
  candidatos: ['pronaca.com.ec', 'pronaca.ec', 'pronaca.com', 'pronaca.net.ec', 'pronacaalimentos.com.ec', 'pronacaalimentos.ec']

CORPORACION FAVORITA C.A.
  razón social: CORPORACION FAVORITA
  comercial: —
  tokens: ['FAVORITA']
  candidatos: ['favorita.com.ec', 'favorita.ec', 'favorita.com', 'favorita.net.ec']

SOCIEDAD NACIONAL DE GALAPAGOS CA
  razón social: SOCIEDAD NACIONAL DE GALAPAGOS
  comercial: —
  tokens: ['GALAPAGOS']
  candidatos: ['galapagos.com.ec', 'galapagos.ec', 'galapagos.com', 'galapagos.net.ec']



## 2. Filtro DNS: no se toca ninguna web todavía

Resolver un nombre de dominio no es visitar el sitio. Este filtro descarta la
mayoría de las candidatas sin generar una sola petición HTTP.

In [3]:
piloto = pd.read_csv(config.ruta_oro(ANIO) / "piloto.csv", dtype={"expediente": int})
muestra = piloto.head(10)
candidatos = pd.DataFrame(
    [{"expediente": e.expediente, "nombre": e.nombre, "dominio": d, "origen": "dominio_derivado"}
     for e in muestra.itertuples() for d in urls.candidatos_de(e.nombre)]
)
vivos = urls.filtrar_por_dns(candidatos)
vivos[["nombre", "dominio"]]

06:03:29  INFO    Dominios que existen: 20 de 60


,nombre,dominio
0,CONSORCIO ECUATORIANO DE TELECOMUNICACIONES S.A. CONECEL,conecel.com.ec
1,CONSORCIO ECUATORIANO DE TELECOMUNICACIONES S.A. CONECEL,conecel.com
2,CONSORCIO ECUATORIANO DE TELECOMUNICACIONES S.A. CONECEL,conecel.net.ec
3,TELCONET S.A.,telconet.ec
4,TELCONET S.A.,telconet.com
5,SERVICIOS DE TELECOMUNICACIONES SETEL S.A,telecomunicaciones.com.ec
6,SERVICIOS DE TELECOMUNICACIONES SETEL S.A,telecomunicaciones.ec
7,SERVICIOS DE TELECOMUNICACIONES SETEL S.A,telecomunicaciones.com
8,PUNTONET S.A.,puntonet.com.ec
9,PUNTONET S.A.,puntonet.ec


## 3. Correr el paso, por tandas

`--limite` controla cuántas empresas por criterio del piloto se trabajan en
cada corrida. El paso es reanudable: lo ya verificado no se vuelve a visitar.

In [4]:
# En un notebook el bucle de eventos ya está corriendo: se usa `await`.
resultado = await urls.ejecutar(ANIO, limite_por_criterio=3)
resultado["url_verificada"].value_counts().to_frame("empresas")

06:03:29  INFO    Buscando la web de 15 empresas


06:03:37  INFO    Dominios que existen: 23 de 100


[2026-11-18 06:03:45]:(verificador_portada) WARNING: Max retries exceeded for blocked request: https://conecel.com/


[2026-11-18 06:03:45]:(verificador_portada) ERROR: Failed after 1 attempts: Failed to perform, curl: (60) SSL: no alternative certificate subject name matches target hostname 'rosado.com'. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.


[2026-11-18 06:03:45]:(verificador_portada) WARNING: Failed to fetch robots.txt for rosado.com: Failed to perform, curl: (60) SSL: no alternative certificate subject name matches target hostname 'rosado.com'. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.; assuming full disallow


[2026-11-18 06:03:46]:(verificador_portada) WARNING: robots.txt for telconet.com returned status 503; assuming full disallow


... (20 outputs left out)

[2026-11-18 06:04:30]:(verificador_portada) WARNING: Failed to fetch robots.txt for saludsa.com.ec: Failed to perform, curl: (28) Connection timed out after 20000 milliseconds. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.; assuming full disallow


06:04:32  INFO    Portadas leídas: 8


06:04:32  INFO    URLs 2025: {'no_encontrada': 9, 'si': 5, 'revisar': 5}


,empresas
url_verificada,
no_encontrada,9
si,5
revisar,5


## 4. El puntaje: qué prueba que el sitio es de esa empresa

| Evidencia | Puntaje |
| --- | --- |
| El RUC aparece en la página | 100 |
| Nombre comercial en el título o el nombre del sitio | 100 |
| Razón social dentro del texto | similitud difusa |
| El dominio coincide con un token del nombre | 70 |

Con `UMBRAL_VERIFICACION` o más se da por verificada; entre
`UMBRAL_REVISION` y ese umbral queda para revisión a mano.

In [5]:
print(f"verificación: ≥ {config.UMBRAL_VERIFICACION}   ·   revisión: ≥ {config.UMBRAL_REVISION}")
verificadas = resultado[resultado["url_verificada"] == "si"]
display(verificadas[["nombre", "url_oficial", "puntaje", "evidencia"]])
resultado["evidencia"].value_counts().to_frame("empresas")

verificación: ≥ 85   ·   revisión: ≥ 60


,nombre,url_oficial,puntaje,evidencia
0,CORPORACION FAVORITA C.A.,https://www.corporacionfavorita.com/,100.0,razon_social
1,PROCESADORA NACIONAL DE ALIMENTOS C.A. PRONACA,https://pronaca.com/,100.0,razon_social
2,FERTISA AGIF C.L.,https://fertisa.com/,100.0,razon_social
3,MINUTOCORP S.A.,https://minutocorp.com/,100.0,razon_social
6,TELCONET S.A.,https://telconet.ec/,100.0,razon_social


,empresas
evidencia,
razon_social,5
dominio_coincide_con_nombre,5


## 5. Revisión a mano

Las filas en `revisar` se corrigen en el CSV poniendo `url_verificada` en
`manual` y la URL correcta: el paso respeta lo corregido y no lo vuelve a
visitar.

In [6]:
pendientes = resultado[resultado["url_verificada"].isin(["revisar", "no_encontrada"])]
print(f"{len(pendientes)} empresas pendientes de revisión")
pendientes[["expediente", "nombre", "url_oficial", "puntaje", "evidencia"]].head(15)

14 empresas pendientes de revisión


,expediente,nombre,url_oficial,puntaje,evidencia
4,47845,CONSORCIO ECUATORIANO DE TELECOMUNICACIONES S.A. CONECEL,NaN,NaN,NaN
5,47972,OTECEL S.A.,NaN,NaN,NaN
7,2913,CORPORACION EL ROSADO S.A.,NaN,NaN,NaN
8,86945,PRIMAX COMERCIAL DEL ECUADOR SOCIEDAD ANONIMA,https://www.primax.com/ec/,70.0,dominio_coincide_con_nombre
9,128168,DINADEC S.A.,NaN,NaN,NaN
10,2151,SOCIEDAD NACIONAL DE GALAPAGOS CA,https://galapagos.com/,70.0,dominio_coincide_con_nombre
11,742,LA FABRIL S.A.,https://fabril.com/,70.0,dominio_coincide_con_nombre
12,4238,NESTLE ECUADOR S.A.,NaN,NaN,NaN
13,740608,"MCKINSEY & COMPANY, INC. ECUADOR",NaN,NaN,NaN
14,704429,IMS-INTERNET MEDIA SERVICES ECUADOR S.A.,NaN,NaN,NaN
